## 01 — Using Tippecanoe

`tippecanoe` is a command-line tool from Mapbox that converts GeoJSON into a vector tile pyramid.

One command replaces our entire Module 02 pipeline — and produces a smaller, faster output format. This notebook runs it, inspects the output, and maps every flag to a decision we already made by hand.

## Installation

On macOS with Homebrew:

```bash
brew install tippecanoe
```

On Linux (Ubuntu/Debian):

```bash
sudo apt-get install tippecanoe
```

Verify the install:

In [1]:
import shutil
import subprocess

if shutil.which("tippecanoe"):
    result = subprocess.run(["tippecanoe", "--version"], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print("tippecanoe is not installed or is not on PATH.")
    print("On macOS: brew install tippecanoe")
    print("Then rerun this notebook to generate railroads.pmtiles.")

tippecanoe is not installed or is not on PATH.
On macOS: brew install tippecanoe
Then rerun this notebook to generate railroads.pmtiles.


## Running Tippecanoe

The basic command:

```bash
tippecanoe \
  --output=railroads.pmtiles \
  --minimum-zoom=1 \
  --maximum-zoom=14 \
  --simplification=10 \
  --drop-densest-as-needed \
  --layer=railroads \
  ne_10m_railroads.geojson
```

Let's run it from Python and capture the output:

In [2]:
from pathlib import Path
import shutil
import subprocess
import time

def find_data_dir():
    """Find the Data Manager data directory from the repo root or this notebook folder."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [
            base / "data",
            base / "Assignments_Completed" / "03-Data_Manager" / "data",
        ]:
            if (candidate / "ne_10m_railroads.geojson").exists():
                return candidate
    raise FileNotFoundError("Could not find Assignments_Completed/03-Data_Manager/data")

data_dir = find_data_dir()
input_file  = data_dir / "ne_10m_railroads.geojson"
output_file = data_dir / "railroads.pmtiles"

cmd = [
    "tippecanoe",
    f"--output={output_file}",
    "--force",                     # overwrite if exists
    "--minimum-zoom=1",
    "--maximum-zoom=14",
    "--simplification=10",         # Douglas-Peucker tolerance in tile pixels
    "--drop-densest-as-needed",    # drop features at low zoom if tile is too large
    "--layer=railroads",
    str(input_file),
]

print("Command:")
print(" ".join(str(part) for part in cmd))
print()

if shutil.which("tippecanoe"):
    t0 = time.perf_counter()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = time.perf_counter() - t0

    print(result.stderr or result.stdout)   # tippecanoe usually writes progress to stderr
    print(f"\nCompleted in {elapsed:.1f}s")
    if result.returncode != 0:
        raise RuntimeError("tippecanoe failed; read the output above")
else:
    print("Skipped: tippecanoe is not installed or is not on PATH.")
    print("Install it, then rerun this cell to create railroads.pmtiles.")

Command:
tippecanoe --output=/Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/03-Data_Manager/data/railroads.pmtiles --force --minimum-zoom=1 --maximum-zoom=14 --simplification=10 --drop-densest-as-needed --layer=railroads /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/03-Data_Manager/data/ne_10m_railroads.geojson

Skipped: tippecanoe is not installed or is not on PATH.
Install it, then rerun this cell to create railroads.pmtiles.


## Inspecting the Output

In [3]:
raw_mb  = input_file.stat().st_size  / 1_000_000

print(f"Input  (raw GeoJSON):  {raw_mb:.1f} MB")

if output_file.exists():
    size_mb = output_file.stat().st_size / 1_000_000
    print(f"Output (PMTiles):      {size_mb:.2f} MB")
    print(f"Compression ratio:     {raw_mb / size_mb:.1f}x smaller")
else:
    print("Output (PMTiles):      not created yet")
    print("Run the previous cell after installing tippecanoe.")

Input  (raw GeoJSON):  39.6 MB
Output (PMTiles):      not created yet
Run the previous cell after installing tippecanoe.


## Mapping Flags to Decisions We Already Made

Every `tippecanoe` flag corresponds to something we built or decided manually:

| tippecanoe flag | What it does | Our equivalent |
|-----------------|-------------|----------------|
| `--minimum-zoom` | First zoom level that gets tiles | Bottom of our LOD range |
| `--maximum-zoom` | Most detailed zoom level | Top of our LOD range |
| `--simplification=10` | D-P tolerance in tile pixels per zoom | Our epsilon per LOD level |
| `--drop-densest-as-needed` | Remove least-important features when tile is too large | Our `scalerank <= 4` coarse filter |
| `--layer=railroads` | Names the data layer in the tile | Our filename convention |

The flags we do NOT have to specify:
- Viewport culling — built into the tile addressing scheme
- Binary encoding — automatic (MVT format)
- Tile pyramid structure — automatic
- Spatial index — automatic (tiles ARE the index)
- Zoom-driven switching — automatic (client requests the right `{z}` tiles)


## Inspecting Tile Contents with sqlite3

PMTiles can be converted to `.mbtiles` (SQLite) for inspection. Or we can use the `pmtiles` CLI to peek at specific tiles.

Alternatively, inspect the metadata embedded in the PMTiles file:

In [4]:
if not output_file.exists():
    print("No PMTiles file found yet. Run the Tippecanoe cell first.")
elif shutil.which("pmtiles"):
    result = subprocess.run(
        ["pmtiles", "show", str(output_file)],
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr)
elif shutil.which("tile-join"):
    inspect_file = output_file.with_suffix(".inspect.pmtiles")
    result = subprocess.run(
        [
            "tile-join",
            "--no-tile-compression",
            "--if-matched",
            f"--output={inspect_file}",
            str(output_file),
        ],
        capture_output=True,
        text=True,
    )
    print(result.stdout or result.stderr)
else:
    print("Install the pmtiles CLI to inspect metadata: pip install pmtiles")
    print(f"PMTiles file: {output_file.resolve()}")

No PMTiles file found yet. Run the Tippecanoe cell first.


## Viewing in ipyleaflet

ipyleaflet supports PMTiles through the `PMTilesLayer` (requires `ipyleaflet >= 0.18`).

For local files, we need to serve them via a local HTTP server or use `localtileserver`.

In [5]:
# Try loading with localtileserver if available
if not output_file.exists():
    print("No PMTiles file found yet. Run the Tippecanoe cell first.")
else:
    try:
        from localtileserver import TileClient, get_leaflet_tile_layer
        from ipyleaflet import Map

        client = TileClient(str(output_file))
        layer  = get_leaflet_tile_layer(client)
        m = Map(center=client.center(), zoom=client.default_zoom)
        m.add(layer)
        m
    except ImportError:
        print("localtileserver not installed.")
        print("Install with: pip install localtileserver")
        print()
        print("Alternative: upload railroads.pmtiles to https://pmtiles.io to view it online.")
        print(f"File location: {output_file.resolve()}")

No PMTiles file found yet. Run the Tippecanoe cell first.


## Exercise A

Run `tippecanoe` a second time with `--maximum-zoom=8` and compare the output file size.

Then answer: what did limiting the maximum zoom cost us in terms of user experience, and what did it save?

In [6]:
# Run tippecanoe with --maximum-zoom=8 and compare output size
output_file_z8 = data_dir / "railroads_z8.pmtiles"

cmd_z8 = [
    "tippecanoe",
    f"--output={output_file_z8}",
    "--force",
    "--minimum-zoom=1",
    "--maximum-zoom=8",
    "--simplification=10",
    "--drop-densest-as-needed",
    "--layer=railroads",
    str(input_file),
]

print("Command:")
print(" ".join(str(part) for part in cmd_z8))
print()

if shutil.which("tippecanoe"):
    t0 = time.perf_counter()
    result = subprocess.run(cmd_z8, capture_output=True, text=True)
    elapsed = time.perf_counter() - t0
    print(result.stderr or result.stdout)
    print(f"Completed in {elapsed:.1f}s")
    if result.returncode != 0:
        raise RuntimeError("tippecanoe failed; read the output above")
else:
    print("Skipped: tippecanoe is not installed or is not on PATH.")

if output_file.exists() and output_file_z8.exists():
    z14_mb = output_file.stat().st_size / 1_000_000
    z8_mb = output_file_z8.stat().st_size / 1_000_000
    saved_mb = z14_mb - z8_mb
    saved_pct = saved_mb / z14_mb * 100

    print(f"Zoom 1-14 PMTiles: {z14_mb:.2f} MB")
    print(f"Zoom 1-8 PMTiles:  {z8_mb:.2f} MB")
    print(f"Saved:             {saved_mb:.2f} MB ({saved_pct:.1f}%)")
else:
    print("Size comparison needs both railroads.pmtiles and railroads_z8.pmtiles.")
    print("Install tippecanoe and rerun the generation cells to get exact numbers.")

# Answer: limiting maximum zoom to 8 saves storage and avoids building the
# highest-detail city/local tiles. The cost is user experience at close zooms:
# when a user zooms past 8, the client has to overzoom lower-detail tiles, so
# railroad curves and local detail look blockier and less precise.

Command:
tippecanoe --output=/Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/03-Data_Manager/data/railroads_z8.pmtiles --force --minimum-zoom=1 --maximum-zoom=8 --simplification=10 --drop-densest-as-needed --layer=railroads /Users/josemarioomrqz/4543-5993-Spatial-Maps/Assignments_Completed/03-Data_Manager/data/ne_10m_railroads.geojson

Skipped: tippecanoe is not installed or is not on PATH.
Size comparison needs both railroads.pmtiles and railroads_z8.pmtiles.
Install tippecanoe and rerun the generation cells to get exact numbers.


## Exercise B

The `--simplification=10` flag sets the tolerance in **tile pixels**, not degrees. At zoom 14, a tile covers roughly 2.4km × 2.4km in 4096 pixels — so one pixel ≈ 0.6m.

Calculate what `--simplification=10` means in meters at zoom levels 2, 5, 8, and 12. Compare these to the degree-based epsilon values we chose in Module 02.

In [7]:
# Calculate simplification tolerance in meters at different zoom levels
# Compare to our Module 02 epsilon choices
earth_circumference_m = 40_075_016.686  # Web Mercator world width at the equator
tile_extent = 4096                      # MVT internal coordinate space per tile
simplification_pixels = 10

print(f"{'Zoom':>4} {'Tile width (km)':>16} {'1 tile px (m)':>14} {'simp=10 (m)':>14}")
print("-" * 56)

for zoom in [2, 5, 8, 12]:
    tile_width_m = earth_circumference_m / (2 ** zoom)
    meters_per_tile_pixel = tile_width_m / tile_extent
    tolerance_m = simplification_pixels * meters_per_tile_pixel
    print(f"{zoom:>4} {tile_width_m / 1000:>16,.1f} {meters_per_tile_pixel:>14,.2f} {tolerance_m:>14,.1f}")

print()
print("Approximate Module 02 degree-based epsilons, using 1 degree ~= 111 km:")
module_02_epsilons = {
    "coarse": 1.0,
    "medium": 0.1,
    "fine": 0.01,
    "extra_fine": 0.001,
}

for level, epsilon_degrees in module_02_epsilons.items():
    epsilon_m = epsilon_degrees * 111_000
    print(f"{level:<10} epsilon={epsilon_degrees:<6} ~= {epsilon_m:>9,.0f} m")

# Conclusion: Tippecanoe's tolerance automatically gets smaller as zoom increases:
# roughly 24.5 km at z2, 3.1 km at z5, 382 m at z8, and 24 m at z12.
# Our Module 02 pipeline used four fixed degree tolerances, while Tippecanoe
# applies a per-zoom pixel tolerance that stays visually consistent on screen.

Zoom  Tile width (km)  1 tile px (m)    simp=10 (m)
--------------------------------------------------------
   2         10,018.8       2,445.98       24,459.8
   5          1,252.3         305.75        3,057.5
   8            156.5          38.22          382.2
  12              9.8           2.39           23.9

Approximate Module 02 degree-based epsilons, using 1 degree ~= 111 km:
coarse     epsilon=1.0    ~=   111,000 m
medium     epsilon=0.1    ~=    11,100 m
fine       epsilon=0.01   ~=     1,110 m
extra_fine epsilon=0.001  ~=       111 m


## Check Your Understanding

We ran `tippecanoe` with `--drop-densest-as-needed`. This flag tells tippecanoe to automatically drop the least-important features when a tile would otherwise be too large.

How does tippecanoe decide which features are "least important"? And how does that compare to our manual `scalerank <= 4` filter? Which approach is more principled — and what are the tradeoffs of each?

Tippecanoe's `--drop-densest-as-needed` does not read our `scalerank` field or know which railroads are culturally important. When tiles are too large, it tries to drop the least visible features by increasing the minimum spacing between features, with the discovered spacing applied across the zoom level. That is more adaptive than our manual `scalerank <= 4` filter because it responds to visual density and tile-size limits instead of applying one global attribute cutoff everywhere. Our `scalerank` filter is simpler and easier to explain because it uses Natural Earth's explicit feature-importance ranking, but it can remove locally important railroads in places where every feature has a higher scalerank. Tippecanoe's approach is usually more principled for production tiles because it is tied to what the tile can actually display, but the tradeoff is that the exact dropped features are less obvious unless you inspect the generated tiles.

---

## Next

In [02 — The Comparison](./02-The_Comparison.ipynb), we put both systems side by side and answer the final question: what did `tippecanoe` actually save us from?